# Lesson 1 — first table · ตารางแรก

LanceDB ไม่มี server ตารางหนึ่งคือ directory หนึ่ง
เขียนหนึ่งครั้ง ได้ไฟล์สามอย่าง `.txn` · `.manifest` · data fragment
บทนี้สร้างตาราง เพิ่มแถว แล้วเปิด directory ดูว่าเกิดอะไรขึ้นจริง

In [1]:
%pip install -q lancedb pandas

Note: you may need to restart the kernel to use updated packages.


`connect` แค่ชี้ไปที่ folder ยังไม่เกิดไฟล์อะไรทั้งนั้น
ไม่มี socket ไม่มี port สอง process เปิด folder เดียวกันได้

In [2]:
import lancedb
import pandas as pd
from pathlib import Path

db = lancedb.connect("./data")

def disk(root: Path):
    """one summary line, then the tree"""
    frags = list((root / "data").glob("*.lance")) if (root / "data").exists() else []
    mans = list((root / "_versions").glob("*.manifest"))
    dels = list((root / "_deletions").glob("*")) if (root / "_deletions").exists() else []
    size = sum(f.stat().st_size for f in root.rglob("*") if f.is_file())
    print(f"fragments={len(frags)} manifests={len(mans)} deletions={len(dels)} bytes={size}")
    def tree(p: Path, prefix=""):
        kids = sorted(p.iterdir(), key=lambda k: (k.is_file(), k.name))
        for i, k in enumerate(kids):
            last = i == len(kids) - 1
            print(prefix + ("└── " if last else "├── ") + k.name)
            if k.is_dir():
                tree(k, prefix + ("    " if last else "│   "))
    tree(root)

ส่ง list ของ dict เข้าไป schema เดาให้เอง
ดูตาราง schema ข้างล่าง type เป็น Arrow type `int64` `string` ไม่ใช่ Python `int` `str`

In [3]:
rows = [
    {"id": 1, "repo": "lance-indexer", "lang": "ts", "stars": 3},
    {"id": 2, "repo": "session-dream", "lang": "ts", "stars": 7},
    {"id": 3, "repo": "arra-memory-py", "lang": "py", "stars": 1},
]
tbl = db.create_table("repos", data=rows, mode="overwrite")
pd.DataFrame([{"column": f.name, "arrow type": str(f.type)} for f in tbl.schema])

[2026-09-10T11:46:49Z WARN  lance::dataset::write::insert] No existing dataset at /opt/Code/github.com/Soul-Brews-Studio/lancedb-oracle/lessons/01-first-table/data/repos.lance, it will be created


,column,arrow type
0,id,int64
1,repo,string
2,lang,string
3,stars,int64


อ่านกลับมาทั้งตาราง 3 แถวเท่าที่ใส่ ลำดับตามที่เขียน

In [4]:
tbl.to_pandas()

,id,repo,lang,stars
0,1,lance-indexer,ts,3
1,2,session-dream,ts,7
2,3,arra-memory-py,py,1


`where` รับ string หน้าตาเหมือน SQL
เงื่อนไขถูกดันลงไปตอนอ่านไฟล์ ไม่ได้อ่านทั้งหมดขึ้นมาแล้วค่อยกรอง
`lang = 'ts'` ควรเหลือ 2 แถว id 1 กับ 2

In [5]:
tbl.search().where("lang = 'ts'").to_pandas()

,id,repo,lang,stars
0,1,lance-indexer,ts,3
1,2,session-dream,ts,7


`add` ไม่แก้ไฟล์เดิม เขียน fragment ใหม่ต่อท้าย แล้วออก manifest ใหม่
ดูตารางข้างล่าง rows 3 → 4 · version 1 → 2

In [6]:
before = {"step": "after create", "rows": tbl.count_rows(), "version": tbl.version}
tbl.add([{"id": 4, "repo": "lanceglass", "lang": "ts", "stars": 2}])
after = {"step": "after add", "rows": tbl.count_rows(), "version": tbl.version}
pd.DataFrame([before, after])

,step,rows,version
0,after create,3,1
1,after add,4,2


เปิด directory ดู บรรทัดแรกคือสรุป บรรทัดถัดไปคือ tree
`create` หนึ่งครั้ง `add` หนึ่งครั้ง ก็ควรเห็น `fragments=2 manifests=2`
แต่ละ fragment คือไฟล์ใน `data/` แต่ละ manifest คือไฟล์ใน `_versions/`

ถ้ารัน notebook นี้ซ้ำ จะเห็นมากกว่านั้น
เพราะ `mode="overwrite"` ไม่ได้ลบของเก่า แค่ออก manifest ใหม่ที่ไม่ชี้ไปหามัน
ไฟล์เก่ายังอยู่ จนกว่าจะสั่ง cleanup เอง (บทที่ 10)

In [7]:
disk(Path("data/repos.lance"))

fragments=2 manifests=2 deletions=0 bytes=3940
├── _transactions
│   ├── 0-4df1e535-d924-496a-b032-89ff335b41fa.txn
│   └── 1-02ae3dee-d95a-4d22-81d4-67414f9b8ef0.txn
├── _versions
│   ├── 18446744073709551613.manifest
│   ├── 18446744073709551614.manifest
│   └── latest_version_hint.json
└── data
    ├── 0100101111010110111110015acd4444179a83978b232df5bb.lance
    └── 100110100100100000111000316d2e44debd971c2be8b232c8.lance
